# QDA vs. Naive Bayes: Flexible Covariances or Independent Features

**DSAI 621 — Week 3.** QDA and naive Bayes both model each class's distribution of $X$ and
classify with Bayes' theorem, and both give each class its own spread. They make opposite bets
about the covariance. QDA estimates a full covariance matrix $\Sigma_k$ for every class, so it
can capture any correlation between predictors, at the price of $p(p+1)/2$ parameters per
class. Gaussian naive Bayes assumes the predictors are independent within each class, so each
$\Sigma_k$ is diagonal and costs only $p$ parameters. In this lab you find out when each bet
pays off, where each one breaks, and which one wins on real data.

All of the code is written for you: run each cell, read what it does, and change it to try
things out. Short exercises marked **E1**, **E2**, … ask you to stop and look at what you just
made, and answer in words.
Exercises marked **T1**, **T2**, … ask you to change a number in a cell, rerun it, and
explain what changed. Each one says which cells to rerun and whether to change the number back
afterwards.

A companion lab, `lda_vs_logistic.ipynb`, runs the same five sections for LDA and logistic
regression.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal, norm
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(621)

### Helper functions

You'll use these four functions in every section; read them, but you don't need to change them.

- `make_gaussian(n, mu0, mu1, Sigma, rng)` draws $n$ observations with equal priors:
  $Y \in \{0, 1\}$, then $X \mid Y = k \sim N(\mu_k, \Sigma_k)$. `Sigma` is either one matrix
  (shared by both classes) or a pair of matrices, one per class.
- `simulation(make_data)` turns a data generator into a *draw* function: each call returns a
  fresh training set of size $n$, plus one large test set that stays fixed.
- `learning_curve(draw, models, ns)` fits every model to `reps` training sets of each size $n$
  and records the average test error. If a model can't be fitted on some training set, its
  error at that $n$ is `NaN` and the `failed` column counts how often.
- `plot_curves(curve)` plots test error against $n$ (log scale), one line style per model.

In [ ]:
def make_gaussian(n, mu0, mu1, Sigma, rng):
    "Equal priors; X | Y=k ~ N(mu_k, Sigma_k). Sigma is one matrix (shared) or a pair."
    Sigmas = Sigma if isinstance(Sigma, (list, tuple)) else (Sigma, Sigma)
    y = rng.integers(0, 2, n)
    X = np.empty((n, len(mu0)))
    for k, (mu, S) in enumerate([(mu0, Sigmas[0]), (mu1, Sigmas[1])]):
        X[y == k] = rng.multivariate_normal(mu, S, size=np.sum(y == k))
    return X, y


def simulation(make_data, n_test=20_000, seed=0):
    "A draw(n, rng) function: a new training set each call, and one fixed test set."
    X_te, y_te = make_data(n_test, np.random.default_rng(seed))
    def draw(n, rng):
        X, y = make_data(n, rng)
        return X, y, X_te, y_te
    return draw


def learning_curve(draw, models, ns, reps=200, seed=1):
    "Average test error of each model over `reps` training sets of each size n."
    rng = np.random.default_rng(seed)
    rows = []
    for n in ns:
        for _ in range(reps):
            X, y, X_te, y_te = draw(n, rng)
            for name, make_model in models.items():
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")   # e.g. convergence warnings on separable data
                    try:
                        err = np.mean(make_model().fit(X, y).predict(X_te) != y_te)
                    except (np.linalg.LinAlgError, ValueError):
                        err = np.nan                  # the model can't be fitted to this sample
                rows.append((name, n, err))
    df = pd.DataFrame(rows, columns=["model", "n", "error"])
    return (df.groupby(["n", "model"])["error"]
              .agg(error=lambda e: e.mean(skipna=False), failed=lambda e: e.isna().sum())
              .reset_index())


STYLES = {"LDA": "o-", "logistic": "s--", "QDA": "^-.", "naive Bayes": "D:"}

def plot_curves(curve, title, bayes=None, ax=None):
    "Test error against n, one line style per model; a dotted line at the Bayes error."
    ax = ax or plt.subplots(figsize=(7, 4))[1]
    for name, d in curve.groupby("model"):
        ax.plot(d["n"], d["error"], STYLES.get(name, "x-"), label=name, lw=1.8, ms=6)
    if bayes is not None:
        ax.axhline(bayes, color="grey", ls=":", label="Bayes error")
    ax.set(xscale="log", xlabel="training size n (log scale)", ylabel="test error", title=title)
    ax.minorticks_off()
    ax.set_xticks(sorted(curve["n"].unique()), labels=sorted(curve["n"].unique()))
    ax.legend()
    return ax


def table(curve):
    "One row per n, one column per model."
    return curve.pivot(index="n", columns="model", values="error").round(3)

---
## 1. Two ways to model a class

Start with two Gaussian classes in two dimensions whose covariances differ, so QDA's
assumption is exactly right and naive Bayes's is wrong (the predictors are correlated within
each class):

$$\mu_0 = (0, 0), \quad \mu_1 = (1, 1.5), \quad \Sigma_0 = \begin{pmatrix} 1 & 0.7 \\ 0.7 & 1 \end{pmatrix}, \quad \Sigma_1 = \begin{pmatrix} 1 & -0.7 \\ -0.7 & 1 \end{pmatrix}, \quad \pi_0 = \pi_1 = 0.5 .$$

Classes are labelled 0 and 1, as scikit-learn does. Fit both methods to one training set and
compare what each estimated for class 0's covariance.

In [ ]:
mu0, mu1 = np.array([0.0, 0.0]), np.array([1.0, 1.5])
Sigma0 = np.array([[1.0, 0.7], [0.7, 1.0]])
Sigma1 = np.array([[1.0, -0.7], [-0.7, 1.0]])

X, y = make_gaussian(100, mu0, mu1, (Sigma0, Sigma1), rng)
qda = QuadraticDiscriminantAnalysis(store_covariance=True).fit(X, y)
nb = GaussianNB().fit(X, y)

print("QDA, class 0 covariance:\n", qda.covariance_[0].round(2))
print("naive Bayes, class 0 covariance (the variances on the diagonal):\n", np.diag(nb.var_[0]).round(2))

Naive Bayes is QDA with every off-diagonal entry of $\Sigma_k$ set to zero. Its density for
class $k$ is a product of one-dimensional normals, so (lecture, §4)

$$\log\big(\pi_k f_k(x)\big) = \log\pi_k + \sum_{j=1}^p \Big[-\tfrac12\log(2\pi\sigma_{kj}^2) - \frac{(x_j - \mu_{kj})^2}{2\sigma_{kj}^2}\Big] .$$

The next cell checks that this is what `GaussianNB` computes. Using `nb.class_prior_`
($\hat\pi_k$), `nb.theta_` (row $k$ holds the $\hat\mu_{kj}$) and `nb.var_` (the
$\hat\sigma_{kj}^2$), it computes $\log(\hat\pi_k\hat f_k(x_i))$ for both classes and every
training point, turns them into $\hat\Pr(Y = 1 \mid x_i)$ with Bayes' theorem, and confirms with
`np.allclose` that they match `nb.predict_proba(X)[:, 1]`.

In [ ]:
def log_joint(x, prior, means, variances):
    "log(pi_k f_k(x)) for every row of x, under independent normal features."
    return np.log(prior) - 0.5 * np.sum(np.log(2 * np.pi * variances) + (x - means) ** 2 / variances, axis=1)

lj = [log_joint(X, nb.class_prior_[k], nb.theta_[k], nb.var_[k]) for k in (0, 1)]
p1 = 1 / (1 + np.exp(lj[0] - lj[1]))          # pi_1 f_1 / (pi_0 f_0 + pi_1 f_1)
print(np.allclose(p1, nb.predict_proba(X)[:, 1]))

Each boundary is the curve where the fitted probability of class 1 is 0.5. The next cell draws
it for both methods, and for the Bayes classifier, which uses the true means and covariances.

In [ ]:
def true_log_odds(x):
    "log[pi_1 f_1(x) / (pi_0 f_0(x))] under the true model of this section (equal priors)."
    return (multivariate_normal(mu1, Sigma1).logpdf(x)
            - multivariate_normal(mu0, Sigma0).logpdf(x))

g1, g2 = np.meshgrid(np.linspace(-3, 4, 300), np.linspace(-3, 4, 300))
grid = np.c_[g1.ravel(), g2.ravel()]
surfaces = {
    "truth (Bayes)": (true_log_odds(grid), 0.0, "-", "black"),
    "QDA": (qda.predict_proba(grid)[:, 1], 0.5, "--", "tab:red"),
    "naive Bayes": (nb.predict_proba(grid)[:, 1], 0.5, ":", "tab:green"),
}

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(*X[y == 0].T, marker="o", facecolors="none", edgecolors="tab:blue", label="class 0")
ax.scatter(*X[y == 1].T, marker="^", color="tab:orange", label="class 1")
for name, (z, level, style, color) in surfaces.items():
    ax.contour(g1, g2, z.reshape(g1.shape), levels=[level], linestyles=style, colors=color, linewidths=2)
    ax.plot([], [], ls=style, color=color, lw=2, label=name)      # legend entry for the contour
ax.set(xlim=(-3, 4), ylim=(-3, 4), xlabel="$x_1$", ylabel="$x_2$", aspect="equal")
ax.legend(loc="upper left")
plt.show()

**E1.** Describe the shape of each boundary. Which one follows the Bayes boundary more closely?
The class-0 cloud is tilted up and to the right, and the class-1 cloud down and to the right:
which of the two methods can "see" that tilt?

<details>
<summary>Show answer: E1</summary>

**E1. Answer.** The Bayes boundary is a hyperbola with two branches. QDA's boundary has the same
shape and follows it fairly closely. Naive Bayes draws a single, gently curved line that misses
the second branch and cuts across the classes at the wrong angle. Only QDA can see the tilt.
The tilt is the correlation between $x_1$ and $x_2$ within a class, an off-diagonal entry of
$\Sigma_k$, and naive Bayes sets those to zero: its fitted class densities have contours that
are ellipses aligned with the axes.

</details>

**T1. Try it.** In the first code cell of this section, change the correlations to $\pm 0.95$: 0.7 to 0.95 in
`Sigma0` and $-0.7$ to $-0.95$ in `Sigma1` (both off-diagonal entries of each). Rerun every cell
in Section 1. Then set all four entries to 0 and rerun again. How does each method's boundary
compare with the Bayes boundary in each case? Set them back to $\pm 0.7$ afterwards.

<details>
<summary>Show answer: T1</summary>

**T1. Answer.** (Each rerun draws a new sample, so details vary.) At $\pm 0.95$ the two clouds
are long, thin and tilted in opposite directions. The Bayes boundary bends sharply around them,
QDA follows it closely, and naive Bayes's boundary is badly wrong, because it can't see the
tilt at all. (On a large test set, QDA's error is typically about 0.06 and naive Bayes's about
0.14, against a Bayes error of 0.06.) At 0, both classes are $N(\mu_k, I)$: the predictors
really are independent and the covariances are equal, so the Bayes boundary is a straight line.
Both fitted boundaries are now close to it and to each other; any curvature they show is
estimation noise. The stronger the within-class correlation, the more naive Bayes's assumption
costs.

</details>

---
## 2. QDA's home turf: correlated predictors, few of them

Average over many training sets from the same model. There's no closed form for this Bayes
error, so estimate it from a large sample using the true log-odds.

In [ ]:
def make_qda_world(n, rng):
    return make_gaussian(n, mu0, mu1, (Sigma0, Sigma1), rng)

X_big, y_big = make_qda_world(500_000, np.random.default_rng(7))
bayes_qda = np.mean((true_log_odds(X_big) > 0) != y_big)

models = {
    "QDA": lambda: QuadraticDiscriminantAnalysis(),
    "naive Bayes": lambda: GaussianNB(),
}
ns = [20, 30, 50, 100, 200, 500]

curve_qda_world = learning_curve(simulation(make_qda_world), models, ns)
plot_curves(curve_qda_world, "Different covariances, correlated predictors (p = 2)", bayes=bayes_qda)
plt.show()
print(f"Bayes error {bayes_qda:.3f}")
table(curve_qda_world)

**E2.** Which method wins, and at which $n$? Does naive Bayes get close to the Bayes error as
$n$ grows? Explain the difference between the two curves in terms of bias and variance.

**E3.** With $p = 2$, how many covariance parameters does each method estimate per class? Why
does QDA's extra flexibility cost so little here, even at $n = 20$?

<details>
<summary>Show answers: E2, E3</summary>

**E2. Answer.** QDA wins at every $n$ where it can be fitted, and by $n = 500$ it is at 0.132,
essentially the Bayes error of 0.131. (At $n = 20$ its entry is `NaN` because at least one of
the 200 training sets had too few points in a class to fit it.) Naive Bayes levels off near
0.154 and never gets close. That gap is bias: naive Bayes's independence assumption is false
here, and more data can't fix a wrong model. QDA's model is right, so its only error above the
Bayes error is variance, which shrinks with $n$.

**E3. Answer.** With $p = 2$, QDA estimates 3 covariance parameters per class (two variances
and one covariance) and naive Bayes estimates 2. QDA pays for only one extra parameter per
class, so its extra variance is tiny, even at $n = 20$. Its extra flexibility is almost free,
and it buys exactly the thing that matters here.

</details>

**T2. Try it.** Make the correlations weaker: in Section 1's first cell, change $\pm 0.7$ to $\pm 0.3$, rerun
that cell and then this section's cell. How big is the gap between the two methods at
$n = 500$ now, compared with $\pm 0.7$? Set the correlations back to $\pm 0.7$ and rerun both
cells afterwards.

<details>
<summary>Show answer: T2</summary>

**T2. Answer.** The gap shrinks from about 0.022 (0.132 against 0.154) to about 0.006 (0.176 against 0.182),
with a Bayes error of 0.174. QDA still wins from $n = 50$ on, but barely. Naive Bayes's bias
comes from the correlations it ignores, so the weaker they are, the less it loses. (The Bayes
error is higher here because the tilted clouds were also easier to tell apart.)

</details>

---
## 3. Naive Bayes's home turf: many independent predictors

Now take $p = 20$ predictors that really are independent within each class: $\Sigma_0 =
\Sigma_1 = I$, with every coordinate of $\mu_1 - \mu_0$ equal to $0.376$. Each predictor on its
own is weak, but together they separate the classes: the means are $\Delta = 0.376\sqrt{20}
\approx 1.68$ standard deviations apart, so the Bayes error is $\Phi(-\Delta/2) \approx 0.20$.

The next cell defines `make_nb_world(n, rng)` and runs the learning curve for both models
with $n$ from 30 to 1000. This time it also prints the `failed` column: how many of the 200
training sets each model couldn't be fitted to.

In [ ]:
p = 20
mu0_20, mu1_20 = np.zeros(p), np.full(p, 0.376)
bayes_20 = norm.cdf(-0.376 * np.sqrt(p) / 2)

def make_nb_world(n, rng):
    return make_gaussian(n, mu0_20, mu1_20, np.eye(p), rng)

curve_nb_world = learning_curve(simulation(make_nb_world), models, [30, 50, 100, 200, 500, 1000])
plot_curves(curve_nb_world, "Independent predictors (p = 20)", bayes=bayes_20)
plt.show()
print(f"Bayes error {bayes_20:.3f}")
print("failed fits out of 200:\n", curve_nb_world.pivot(index="n", columns="model", values="failed"))
table(curve_nb_world)

**E4.** At $n = 30$, QDA fails on every training set, and at $n = 50$ on some. Why? (How many
observations does each class have, and what is the rank of a sample covariance matrix computed
from $n_k$ points in $p = 20$ dimensions?)

**E5.** Count the parameters each method estimates in this world: means, variances and
covariances, for both classes. How does the ratio help explain the gap between the two curves
at $n = 100$? What happens to the gap by $n = 1000$?

<details>
<summary>Show answers: E4, E5</summary>

**E4. Answer.** At $n = 30$ each class has about 15 observations. A sample covariance matrix
computed from $n_k$ points has rank at most $n_k - 1$, so with $n_k - 1 < p = 20$ it is singular:
$\hat\Sigma_k^{-1}$ and $\log|\hat\Sigma_k|$ don't exist. QDA needs $n_k \ge 21$ in *both*
classes. At $n = 50$ the classes average 25 points each, but in 42 of the 200 training sets one
class had too few.

**E5. Answer.** Per class, QDA estimates 20 means, 20 variances and $20 \cdot 19 / 2 = 190$
covariances, so 460 parameters for both classes. Naive Bayes estimates 20 means and 20 variances
per class, so 80. At $n = 100$ QDA estimates more than four parameters per observation, and its
test error (0.382) is close to guessing, while naive Bayes (0.255) is already within 0.06 of
the Bayes error. By $n = 1000$ the gap has shrunk to 0.233 against 0.206, but QDA is still
behind. Notice that QDA's model is also *correct* here ($\Sigma_k = I$ is a special case of a
full covariance), so all of its extra error is variance.

</details>

**T3. Try it.** In this section's cell, change `p = 20` to `p = 5` and rerun it (the mean difference per
predictor stays 0.376, so the Bayes error changes too). Does QDA still fail? How do the two
methods compare now? Set it back to 20 afterwards.

<details>
<summary>Show answer: T3</summary>

**T3. Answer.** QDA never fails, since each class has far more than 6 observations even at $n = 30$. Naive
Bayes still wins at every $n$, because its assumption is exactly true, but the gap is much
smaller: 0.388 against 0.370 at $n = 100$ and 0.344 against 0.342 at $n = 1000$, with a Bayes
error of 0.337. With $p = 5$, QDA estimates $2(5 + 5 + 10) = 40$ parameters to naive Bayes's 20,
a ratio of 2 rather than almost 6, so its extra variance is much smaller. Naive Bayes's
advantage grows with $p$.

</details>

---
## 4. A breaking point: redundant predictors

Naive Bayes multiplies one density per predictor, so it counts each predictor's evidence once.
What if two predictors carry the *same* evidence? Take one informative predictor,
$X_1 \mid Y = k \sim N(1.5k, 1)$, and add exact copies of it. The copies add no information, so
the Bayes classifier and its error don't change.

The next cell does this for 0, 1, 3 and 9 copies, building the matrices with
`np.repeat(x, copies + 1, axis=1)`. For each, it fits naive Bayes and records its test error,
its test log loss (`log_loss(y_te, m.predict_proba(X_te))`), which measures how good the
probabilities are, and the fraction of test points where $\hat\Pr(Y = 1 \mid x)$ is above 0.99
or below 0.01. Then it tries to fit QDA with 1 copy.

In [ ]:
def make_one_feature(n, rng):
    y = rng.integers(0, 2, n)
    return (rng.standard_normal(n) + 1.5 * y)[:, None], y

x_tr, y_tr = make_one_feature(200, np.random.default_rng(8))
x_te, y_te = make_one_feature(20_000, np.random.default_rng(9))

rows = []
for copies in [0, 1, 3, 9]:
    X_tr, X_te = np.repeat(x_tr, copies + 1, axis=1), np.repeat(x_te, copies + 1, axis=1)
    m = GaussianNB().fit(X_tr, y_tr)
    prob = m.predict_proba(X_te)[:, 1]
    rows.append({"copies": copies, "test error": 1 - m.score(X_te, y_te),
                 "test log loss": log_loss(y_te, m.predict_proba(X_te)),
                 "share of probabilities < 0.01 or > 0.99": np.mean((prob < 0.01) | (prob > 0.99))})
display(pd.DataFrame(rows).set_index("copies").round(3))

try:
    QuadraticDiscriminantAnalysis().fit(np.repeat(x_tr, 2, axis=1), y_tr)
except np.linalg.LinAlgError as err:
    print("QDA with one copy:", err)

**E6.** The test error barely changes as you add copies, but the log loss climbs. Write naive
Bayes's log-odds with $c$ copies in terms of its log-odds with none. Why does the classification
stay the same while the probabilities get more extreme? (Assume the estimated priors are equal.)

**E7.** Why can't QDA be fitted with even one copy? Real predictors are rarely exact copies,
but they are often highly correlated. What does this section suggest about each method when
predictors are strongly correlated?

<details>
<summary>Show answers: E6, E7</summary>

**E6. Answer.** Each copy contributes exactly the same factor $\hat f_{1j}(x_1)/\hat f_{0j}(x_1)$
to naive Bayes's product, so with $c$ copies

$$\log\frac{\hat\Pr(Y = 1 \mid x)}{\hat\Pr(Y = 0 \mid x)} = \log\frac{\hat\pi_1}{\hat\pi_0} + (c + 1)\log\frac{\hat f_1(x_1)}{\hat f_0(x_1)} .$$

With equal priors the first term is zero, so multiplying the log-odds by $c + 1$ doesn't
change its sign, and the classification is the same. (The tiny changes in test error come from
$\hat\pi_0 \ne \hat\pi_1$.) The magnitude is multiplied by $c + 1$, though, so the probabilities
are pushed toward 0 and 1. With 9 copies, 81% of test probabilities are below 0.01 or above
0.99, compared with 0.6% with none, and the log loss rises from 0.48 to 1.93. Naive Bayes counts
the same evidence ten times.

**E7. Answer.** With an exact copy, the two columns are identical, so $\hat\Sigma_k$ is singular
(its rank is 1, not 2). The QDA discriminant needs $\hat\Sigma_k^{-1}$ and $\log|\hat\Sigma_k|$,
which don't exist, so scikit-learn refuses to fit. With strongly correlated predictors, the
two methods fail in different ways. Naive Bayes still runs, but it double-counts shared
evidence, so its probabilities are overconfident even when its classifications are fine. QDA's
$\hat\Sigma_k$ is nearly singular, so its inverse is unstable and the fit has high variance or
fails outright. That's why Section 5 drops perimeter and area, which are almost functions of
radius.

</details>

**T4. Try it.** In `make_one_feature`, make class 1 rare: change `y = rng.integers(0, 2, n)` to
`y = (rng.random(n) < 0.2).astype(int)`, and rerun the cell. Does the test error still stay
the same as you add copies? Why is this different from E6? Change it back afterwards.

<details>
<summary>Show answer: T4</summary>

**T4. Answer.** No: the test error now climbs, from 0.151 with no copies to 0.167, 0.187 and 0.207 with 1, 3 and
9 copies (the log loss rises too). The estimated priors are now about 0.78 and 0.22, so the
prior term $\log(\hat\pi_1/\hat\pi_0)$ is no longer 0. Each copy counts the predictor's evidence
again, and with enough copies that evidence overwhelms the prior: naive Bayes ends up ignoring
how rare class 1 is and calls too many points class 1. With equal priors there was no prior
term to overwhelm, so only the probabilities changed.

</details>

---
## 5. Real data: breast cancer biopsies

The Wisconsin breast cancer data (Street, Wolberg and Mangasarian, 1993) ship with scikit-learn.
For 569 tumours, they record 30 measurements of the cell nuclei in a biopsy image, and whether
the tumour was malignant (212) or benign (357). We use the 8 "mean" measurements, leaving out
perimeter and area: they are almost exact functions of radius (correlation above 0.99), and
nearly collinear predictors make a covariance matrix close to singular.

Nothing here is simulated, so there's no Bayes error and no fixed test set. Instead, each
repetition draws a random training set of size $n$ and tests on the other $569 - n$ tumours.
We standardize inside a pipeline, so the scaling is learned from the training set only.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
features = [c for c in cancer.data.columns[:10] if c not in ("mean perimeter", "mean area")]
X_bc = cancer.data[features].to_numpy()
y_bc = 1 - cancer.target.to_numpy()          # 1 = malignant, 0 = benign

def draw_cancer(n, rng):
    "A random training set of size n; the rest of the data is the test set."
    idx = rng.permutation(len(y_bc))
    return X_bc[idx[:n]], y_bc[idx[:n]], X_bc[idx[n:]], y_bc[idx[n:]]

print(features)
print(f"{len(y_bc)} tumours, {y_bc.sum()} malignant")

The next cell runs a learning curve on `draw_cancer` for QDA and naive Bayes, each in a
pipeline after `StandardScaler()`, with $n$ = 50, 100, 200 and 400 and 200 repetitions, and
plots it with the table.

In [ ]:
models_bc = {
    "QDA": lambda: make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis()),
    "naive Bayes": lambda: make_pipeline(StandardScaler(), GaussianNB()),
}
curve_bc = learning_curve(draw_cancer, models_bc, [50, 100, 200, 400])
plot_curves(curve_bc, "Breast cancer data, 8 predictors")
plt.show()
table(curve_bc)

**E8.** Which method wins at small $n$, and which at large $n$? Which of Sections 2 and 3 does
each end of the curve resemble? The next cell starts with the correlations between the
predictors within the benign class: are they independent?

To finish, put all four of this week's methods side by side on the same data.

In [ ]:
display(pd.DataFrame(np.corrcoef(X_bc[y_bc == 0].T), index=features, columns=features).round(2))

models_all = models_bc | {
    "LDA": lambda: make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=10_000)),
}
curve_all = learning_curve(draw_cancer, models_all, [50, 100, 200, 400])
plot_curves(curve_all, "Breast cancer data: all four methods")
plt.show()
table(curve_all)

**E9.** Which method is best at each $n$? Is any one of them best everywhere? For each method,
name the assumption that explains where it does well or badly.

<details>
<summary>Show answers: E8, E9</summary>

**E8. Answer.** Naive Bayes wins at $n = 50$ (0.090 against 0.115). QDA wins from $n = 100$ on,
and by a wide margin at $n = 400$ (0.058 against 0.081). The small-$n$ end looks like Section 3:
QDA estimates 88 parameters from 50 tumours, and naive Bayes's low variance wins. The large-$n$
end looks like Section 2: the predictors are strongly correlated within a class (compactness,
concavity and concave points have correlations of 0.7 to 0.75; fractal dimension and radius
$-0.53$), so naive Bayes's model is wrong and its error stops improving near 0.08, while QDA
keeps improving.

**E9. Answer.** Naive Bayes is best at $n = 50$, narrowly ahead of LDA. QDA is best at
$n = 100$ and clearly best at 200 and 400, with logistic regression second. No method is best
everywhere.

- *Naive Bayes:* the fewest parameters, so the best at small $n$, but its independence
  assumption is badly wrong, so it is the worst at large $n$.
- *LDA:* a moderate number of parameters, but its *shared* covariance is wrong (malignant
  tumours are much more variable), so it levels off around 0.073.
- *Logistic regression:* no assumption about the distribution of $X$, so it keeps improving
  with $n$, but its boundary is linear.
- *QDA:* the most flexible. A class-specific covariance and a curved boundary suit these data,
  and it wins once $n$ is large enough to pay for its parameters.

</details>

**T5. Try it.** Put perimeter and area back: in the cell that loads the breast cancer data, change
`("mean perimeter", "mean area")` to `()`, and rerun it and the QDA and naive Bayes learning
curve cell. What happens to each method? Change it back and rerun both cells afterwards.

<details>
<summary>Show answer: T5</summary>

**T5. Answer.** QDA's errors become `NaN` at every $n$: it can't be fitted on almost any training set (196 of
200 at $n = 50$ and all 200 at $n = 400$). Radius, perimeter and area are almost exact functions
of each other, so within a class the sample covariance matrix is numerically singular, just
like the exact copies in Section 4. Naive Bayes barely notices: its error stays at about
0.085. It never inverts a covariance matrix, so collinearity can't break it, although it now
counts the size information three times.

</details>

---
## 6. Wrap-up questions

**Q1.** QDA and naive Bayes both fit a Gaussian to each class. In two or three sentences,
explain the difference in what they assume, and how it played out in Sections 2 and 3.

**Q2.** Section 4 showed that naive Bayes can be right about the class and badly wrong about
the probability. Give an application where that matters, and one where it doesn't.

**Q3.** The lecture's "Wins when" row says QDA wins when the covariances differ and $n \gg p$,
and naive Bayes when $p$ is large and $n$ is small. Which of your results support that, and is
there any result that it doesn't predict?

<details>
<summary>Show answers: Q1, Q2, Q3</summary>

**Q1. Answer.** Both model each class as a Gaussian with its own covariance. QDA lets
$\Sigma_k$ be any covariance matrix; naive Bayes requires it to be diagonal, that is, the
predictors independent within a class. In Section 2 the predictors were correlated, so naive
Bayes was biased and QDA reached the Bayes error. In Section 3 independence held and $p = 20$, so
QDA's 460 parameters gave it huge variance (or made it impossible to fit), and naive Bayes was
close to the Bayes error.

**Q2. Answer.** Probabilities matter whenever a decision uses their size and not just which
side of 0.5 they fall on. For example, a hospital might send patients for a biopsy when the
risk of malignancy is above 5%, or an insurer might price policies by expected cost. An
overconfident 0.999 would distort both. They don't matter when only the predicted class is
used, as in a spam filter with a fixed 0.5 cutoff. Multiplying the log-odds by a constant also
keeps the *ranking* of cases unchanged, so a "review the top 100" rule is unaffected.

**Q3. Answer.** Section 2 (covariances differ, $n \gg p = 2$: QDA wins), Section 3 ($p = 20$ and
small $n$: naive Bayes wins) and the breast cancer crossover all fit it. Two results go further
than the rule. In Section 3 naive Bayes is still ahead at $n = 1000$, where $n \gg p$, because its
assumption is exactly true there. And in Section 2 QDA wins even at $n = 20$, which isn't $n \gg p$
in absolute terms, because with $p = 2$ it has only one extra parameter per class. The rule of
thumb is about variance; it says nothing about whether naive Bayes's independence assumption is
true.

</details>